# Notebook 08 — Train-Only Probability and Signal Policy

This implementation deliberately does **not** open the unseen test.

It freezes:

- whether the Stage 07 XGBoost output can be calibrated;
- how raw scores may be interpreted;
- how same-day candidate events are ranked;
- how many daily candidates become decision-support signals.

The repository keeps the legacy filename `08_unseen_test_evaluation.ipynb`,
but unseen-test evaluation is deferred until this policy is audited and frozen.


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import yaml

print("Python:", sys.version.split()[0])
print("numpy:", np.__version__)
print("pandas:", pd.__version__)


## 1. Locate the repository and import project modules

In [ ]:
def locate_repository_root(start: Path) -> Path:
    current = start.resolve()

    for candidate in [current, *current.parents]:
        if (
            (candidate / "configs").exists()
            and (candidate / "notebooks").exists()
            and (candidate / "src").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Repository root was not found. Open the project root in VS Code."
    )


REPOSITORY_ROOT = locate_repository_root(Path.cwd())

if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from src.evaluation.probability_policy import (
    METHOD_COMPLEXITY_ORDER,
    fit_probability_calibrator,
    is_non_decreasing_mapping,
    predict_probability_calibrator,
    probability_metrics,
)
from src.evaluation.signal_policy import (
    evaluate_signal_policy_by_fold,
    select_daily_top_fraction,
    select_signal_policy_hierarchically,
    summarize_signal_policy,
)
from src.utils.paths import repository_result_paths
from src.utils.reproducibility import (
    git_commit_sha,
    software_manifest,
    stable_object_hash,
)

print("Repository root:", REPOSITORY_ROOT)


## 2. Load configuration and frozen lineage

In [ ]:
def load_yaml(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as file_obj:
        value = yaml.safe_load(file_obj)

    if not isinstance(value, dict):
        raise ValueError(f"Configuration must be a mapping: {path}")

    return value


def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as file_obj:
        value = json.load(file_obj)

    if not isinstance(value, dict):
        raise ValueError(f"JSON artifact must be an object: {path}")

    return value


paths_config = load_yaml(REPOSITORY_ROOT / "configs" / "paths.yaml")
policy_config = load_yaml(
    REPOSITORY_ROOT / "configs" / "signal_policy.yaml"
)
RESULT_PATHS = repository_result_paths(REPOSITORY_ROOT, paths_config)

prediction_path = (
    RESULT_PATHS["predictions"]
    / "06_walk_forward_validation_predictions.csv"
)
stage06_decision_path = (
    RESULT_PATHS["manifests"] / "06_model_selection_decision.json"
)
stage06_manifest_path = (
    RESULT_PATHS["manifests"]
    / "06_optuna_model_selection_manifest.json"
)
stage07_manifest_path = (
    RESULT_PATHS["manifests"]
    / "07_frozen_model_training_manifest.json"
)

for path in [
    prediction_path,
    stage06_decision_path,
    stage06_manifest_path,
    stage07_manifest_path,
]:
    if not path.exists():
        raise FileNotFoundError(f"Required frozen artifact is missing: {path}")

stage06_decision = load_json(stage06_decision_path)
stage06_manifest = load_json(stage06_manifest_path)
stage07_manifest = load_json(stage07_manifest_path)

assert policy_config["status"] == "stage_08_configured_v1"
assert stage06_decision["primary_selected_model"] == "xgboost"
assert stage06_decision["threshold_selected"] is False
assert stage06_decision["unseen_test_used"] is False
assert stage06_manifest["status"] == "frozen_after_integrity_checks"
assert stage06_manifest["unseen_test_used"] is False
assert stage07_manifest["status"] == "frozen_after_integrity_checks"
assert stage07_manifest["model"]["selected_model"] == "xgboost"
assert stage07_manifest["probability_policy"]["calibration_fitted"] is False
assert stage07_manifest["probability_policy"]["threshold_selected"] is False
assert stage07_manifest["unseen_test_used"] is False

print("Stage 06 code SHA:", stage06_manifest["git_commit_sha"])
print("Stage 07 code SHA:", stage07_manifest["git_commit_sha"])
print("Stage 07 model SHA256:", stage07_manifest["model"]["model_sha256"])


## 3. Load and audit selected-model OOF predictions

In [ ]:
prediction_columns = [
    "model_name",
    "fold_id",
    "event_id",
    "symbol",
    "dEven",
    "event_end_date",
    "meta_label",
    "probability_positive",
    "validation_metric_weight",
]

all_oof = pd.read_csv(
    prediction_path,
    usecols=prediction_columns,
    low_memory=False,
)

selected_model = str(policy_config["input"]["selected_model"])
oof = all_oof.loc[
    all_oof["model_name"].eq(selected_model)
].copy()

oof["fold_id"] = pd.to_numeric(
    oof["fold_id"],
    errors="raise",
).astype(int)
oof["dEven"] = pd.to_datetime(
    oof["dEven"],
    errors="raise",
).dt.normalize()
oof["event_end_date"] = pd.to_datetime(
    oof["event_end_date"],
    errors="raise",
).dt.normalize()
oof["meta_label"] = pd.to_numeric(
    oof["meta_label"],
    errors="raise",
).astype(int)
oof["probability_positive"] = pd.to_numeric(
    oof["probability_positive"],
    errors="raise",
).astype(float)
oof["validation_metric_weight"] = pd.to_numeric(
    oof["validation_metric_weight"],
    errors="raise",
).astype(float)

oof = oof.sort_values(
    ["fold_id", "dEven", "symbol", "event_id"],
    kind="stable",
).reset_index(drop=True)

expected_rows = int(policy_config["input"]["expected_oof_rows"])
expected_fold_counts = {
    int(key): int(value)
    for key, value in policy_config["input"]["expected_fold_counts"].items()
}
actual_fold_counts = (
    oof.groupby("fold_id").size().astype(int).to_dict()
)

assert len(oof) == expected_rows
assert actual_fold_counts == expected_fold_counts
assert oof["event_id"].is_unique
assert oof["meta_label"].isin([0, 1]).all()
assert np.isfinite(
    oof["probability_positive"].to_numpy(dtype=float)
).all()
assert oof["probability_positive"].between(0.0, 1.0).all()
assert oof["validation_metric_weight"].eq(1.0).all()
assert set(oof["fold_id"]) == {1, 2, 3, 4, 5}

oof_input_audit_df = (
    oof.groupby("fold_id", as_index=False)
    .agg(
        events=("event_id", "size"),
        dates=("dEven", "nunique"),
        first_signal_date=("dEven", "min"),
        last_signal_date=("dEven", "max"),
        positive_fraction=("meta_label", "mean"),
        minimum_raw_score=("probability_positive", "min"),
        mean_raw_score=("probability_positive", "mean"),
        maximum_raw_score=("probability_positive", "max"),
    )
)
oof_input_audit_df["selected_model"] = selected_model
oof_input_audit_df["duplicate_event_ids"] = 0
oof_input_audit_df["nonfinite_scores"] = 0
oof_input_audit_df["unseen_test_used"] = False

oof_input_audit_df.to_csv(
    RESULT_PATHS["audits"] / "08_oof_input_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

display(oof_input_audit_df)


## 4. Expanding-origin calibration evaluation

For target fold `k`, sigmoid and isotonic mappings are fitted only on OOF rows
from earlier folds. Fold 5 remains an internal confirmation fold and is not used
to choose the calibration method.


In [ ]:
calibration_config = policy_config["calibration_evaluation"]
candidate_methods = list(calibration_config["candidate_methods"])
target_folds = [
    int(value)
    for value in calibration_config["expanding_origin_target_folds"]
]
development_calibration_folds = {
    int(value)
    for value in calibration_config["development_folds"]
}
confirmation_fold = int(calibration_config["confirmation_fold"])
clip = float(calibration_config["probability_clip"])
ece_bins = int(calibration_config["ece_bins"])
seed = int(stage07_manifest["model"]["random_seed"])

calibration_rows = []
calibration_prediction_parts = []

for target_fold in target_folds:
    prior = oof.loc[oof["fold_id"] < target_fold].copy()
    target = oof.loc[oof["fold_id"] == target_fold].copy()

    if prior.empty or target.empty:
        raise RuntimeError(
            f"Expanding-origin calibration data missing for fold {target_fold}."
        )

    raw_metrics = probability_metrics(
        target["meta_label"].to_numpy(dtype=int),
        target["probability_positive"].to_numpy(dtype=float),
        ece_bins=ece_bins,
    )

    for method in candidate_methods:
        calibrator = fit_probability_calibrator(
            method,
            prior["probability_positive"].to_numpy(dtype=float),
            prior["meta_label"].to_numpy(dtype=int),
            seed=seed,
            clip=clip,
        )
        mapped = predict_probability_calibrator(
            calibrator,
            target["probability_positive"].to_numpy(dtype=float),
            clip=clip,
        )
        metrics = probability_metrics(
            target["meta_label"].to_numpy(dtype=int),
            mapped,
            ece_bins=ece_bins,
        )

        monotonic = is_non_decreasing_mapping(
            target["probability_positive"].to_numpy(dtype=float),
            mapped,
        )

        row = {
            "target_fold": int(target_fold),
            "method": method,
            "calibration_train_folds": ",".join(
                str(value)
                for value in sorted(prior["fold_id"].unique())
            ),
            "calibration_train_events": int(len(prior)),
            "target_events": int(len(target)),
            "mapping_non_decreasing": bool(monotonic),
            "mapping_direction": calibrator.metadata.get(
                "monotonic_direction"
            ),
            "sigmoid_slope": calibrator.metadata.get("slope"),
            "sigmoid_intercept": calibrator.metadata.get("intercept"),
            "isotonic_threshold_count": calibrator.metadata.get(
                "threshold_count"
            ),
            **metrics,
            "auc_drop_vs_raw": float(
                raw_metrics["roc_auc"] - metrics["roc_auc"]
            ),
            "brier_improvement_vs_raw": float(
                raw_metrics["brier_score"] - metrics["brier_score"]
            ),
            "logloss_improvement_vs_raw": float(
                raw_metrics["log_loss"] - metrics["log_loss"]
            ),
            "used_for_method_selection": (
                target_fold in development_calibration_folds
            ),
            "confirmation_only": target_fold == confirmation_fold,
            "unseen_test_used": False,
        }
        calibration_rows.append(row)

        prediction_part = target[
            [
                "fold_id",
                "event_id",
                "symbol",
                "dEven",
                "meta_label",
                "probability_positive",
            ]
        ].copy()
        prediction_part["calibration_method"] = method
        prediction_part["mapped_score"] = mapped
        calibration_prediction_parts.append(prediction_part)

calibration_fold_metrics_df = pd.DataFrame(calibration_rows)
calibration_predictions_df = pd.concat(
    calibration_prediction_parts,
    ignore_index=True,
)

calibration_fold_metrics_df.to_csv(
    RESULT_PATHS["audits"]
    / "08_calibration_temporal_fold_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

display(calibration_fold_metrics_df)


## 5. Select or reject post-hoc calibration

In [ ]:
development_calibration = calibration_fold_metrics_df.loc[
    calibration_fold_metrics_df["target_fold"].isin(
        development_calibration_folds
    )
].copy()

raw_development = development_calibration.loc[
    development_calibration["method"].eq("raw_identity")
]
raw_mean_brier = float(raw_development["brier_score"].mean())
raw_mean_logloss = float(raw_development["log_loss"].mean())

calibration_summary_rows = []

for method, method_frame in development_calibration.groupby(
    "method",
    sort=True,
):
    mean_brier = float(method_frame["brier_score"].mean())
    mean_logloss = float(method_frame["log_loss"].mean())

    mapping_admissible = bool(
        method_frame["mapping_non_decreasing"].all()
    )
    auc_admissible = bool(
        method_frame["auc_drop_vs_raw"].max()
        <= float(calibration_config["maximum_allowed_auc_drop"])
        + 1.0e-12
    )
    improvement_admissible = bool(
        (
            raw_mean_brier - mean_brier
            >= float(
                calibration_config[
                    "minimum_mean_brier_improvement"
                ]
            )
        )
        or (
            raw_mean_logloss - mean_logloss
            >= float(
                calibration_config[
                    "minimum_mean_logloss_improvement"
                ]
            )
        )
    )

    if method == "raw_identity":
        admissible = True
        improvement_admissible = True
    else:
        admissible = bool(
            mapping_admissible
            and auc_admissible
            and improvement_admissible
        )

    calibration_summary_rows.append(
        {
            "method": method,
            "development_folds": ",".join(
                str(value)
                for value in sorted(development_calibration_folds)
            ),
            "mean_brier_score": mean_brier,
            "worst_fold_brier_score": float(
                method_frame["brier_score"].max()
            ),
            "mean_log_loss": mean_logloss,
            "worst_fold_log_loss": float(
                method_frame["log_loss"].max()
            ),
            "mean_expected_calibration_error": float(
                method_frame["expected_calibration_error"].mean()
            ),
            "mean_roc_auc": float(method_frame["roc_auc"].mean()),
            "worst_auc_drop_vs_raw": float(
                method_frame["auc_drop_vs_raw"].max()
            ),
            "mean_brier_improvement_vs_raw": float(
                raw_mean_brier - mean_brier
            ),
            "mean_logloss_improvement_vs_raw": float(
                raw_mean_logloss - mean_logloss
            ),
            "all_mappings_non_decreasing": mapping_admissible,
            "auc_admissible": auc_admissible,
            "improvement_admissible": improvement_admissible,
            "admissible": admissible,
            "complexity_order": int(
                METHOD_COMPLEXITY_ORDER[method]
            ),
        }
    )

calibration_method_summary_df = pd.DataFrame(
    calibration_summary_rows
)

admissible_calibration = calibration_method_summary_df.loc[
    calibration_method_summary_df["admissible"]
].copy()

if admissible_calibration.empty:
    raise RuntimeError("No admissible calibration policy exists.")

admissible_calibration = admissible_calibration.sort_values(
    [
        "mean_brier_score",
        "worst_fold_brier_score",
        "mean_log_loss",
        "complexity_order",
    ],
    ascending=[True, True, True, True],
    kind="stable",
)

selected_calibration_method = str(
    admissible_calibration.iloc[0]["method"]
)
calibration_method_summary_df["selected"] = (
    calibration_method_summary_df["method"]
    .eq(selected_calibration_method)
)

calibration_method_summary_df = (
    calibration_method_summary_df.sort_values(
        [
            "selected",
            "admissible",
            "mean_brier_score",
            "worst_fold_brier_score",
            "mean_log_loss",
            "complexity_order",
        ],
        ascending=[False, False, True, True, True, True],
        kind="stable",
    ).reset_index(drop=True)
)

expected_calibration_method = str(
    calibration_config["expected_selected_method"]
)
assert selected_calibration_method == expected_calibration_method
assert selected_calibration_method == "raw_identity"

calibration_fitted = selected_calibration_method != "raw_identity"

calibration_confirmation_df = calibration_fold_metrics_df.loc[
    (
        calibration_fold_metrics_df["target_fold"]
        .eq(confirmation_fold)
    )
].copy()

calibration_method_summary_df.to_csv(
    RESULT_PATHS["audits"]
    / "08_calibration_method_summary.csv",
    index=False,
    encoding="utf-8-sig",
)
calibration_confirmation_df.to_csv(
    RESULT_PATHS["audits"]
    / "08_calibration_confirmation.csv",
    index=False,
    encoding="utf-8-sig",
)

display(calibration_method_summary_df)
display(calibration_confirmation_df)

print("Selected calibration method:", selected_calibration_method)
print("Probability calibrator fitted:", calibration_fitted)
print(
    "Interpretation: raw model output is retained as an "
    "uncalibrated ranking score."
)


## 6. Evaluate candidate daily cross-sectional signal fractions

Signal selection uses the raw score ranking, so it does not depend on an
unstable probability interpretation.


In [ ]:
signal_config = policy_config["signal_policy"]
candidate_fractions = [
    float(value)
    for value in signal_config["candidate_fractions"]
]
development_signal_folds = {
    int(value)
    for value in signal_config["development_folds"]
}
signal_confirmation_fold = int(signal_config["confirmation_fold"])
minimum_per_date = int(
    signal_config["minimum_signals_per_date"]
)

policy_fold_parts = []
policy_summary_rows = []
policy_prediction_parts = []

for fraction in candidate_fractions:
    selected_frame = select_daily_top_fraction(
        oof,
        score_column="probability_positive",
        date_column="dEven",
        fraction=fraction,
        minimum_per_date=minimum_per_date,
        symbol_column="symbol",
        event_id_column="event_id",
    )
    selected_frame["signal_fraction"] = fraction

    fold_metrics = evaluate_signal_policy_by_fold(
        selected_frame,
        fold_column="fold_id",
        label_column="meta_label",
        selected_column="selected_signal",
        date_column="dEven",
    )
    fold_metrics["signal_fraction"] = fraction
    fold_metrics["development_fold"] = (
        fold_metrics["fold_id"].isin(development_signal_folds)
    )
    fold_metrics["confirmation_only"] = (
        fold_metrics["fold_id"].eq(signal_confirmation_fold)
    )

    policy_fold_parts.append(fold_metrics)
    policy_prediction_parts.append(selected_frame)

    policy_summary_rows.append(
        summarize_signal_policy(
            fold_metrics,
            fraction=fraction,
            development_folds=development_signal_folds,
        )
    )

signal_policy_fold_metrics_df = pd.concat(
    policy_fold_parts,
    ignore_index=True,
)
signal_policy_summary_df = pd.DataFrame(policy_summary_rows)

selected_policy_row, signal_policy_ranking_df = (
    select_signal_policy_hierarchically(
        signal_policy_summary_df,
        minimum_signals_per_fold=int(
            signal_config[
                "minimum_signals_per_development_fold"
            ]
        ),
        minimum_worst_precision_lift=float(
            signal_config[
                "minimum_worst_fold_precision_lift"
            ]
        ),
        near_best_tolerance=float(
            signal_config[
                "near_best_worst_precision_lift_tolerance"
            ]
        ),
    )
)

selected_signal_fraction = float(
    selected_policy_row["signal_fraction"]
)
expected_signal_fraction = float(
    signal_config["expected_selected_fraction"]
)
assert np.isclose(
    selected_signal_fraction,
    expected_signal_fraction,
    atol=1.0e-12,
    rtol=0.0,
)

signal_policy_fold_metrics_df.to_csv(
    RESULT_PATHS["audits"]
    / "08_signal_policy_fold_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)
signal_policy_ranking_df.to_csv(
    RESULT_PATHS["audits"]
    / "08_signal_policy_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

display(signal_policy_ranking_df)


## 7. Apply the selected rule and hold out fold 5 for confirmation

In [ ]:
selected_oof_policy_df = select_daily_top_fraction(
    oof,
    score_column="probability_positive",
    date_column="dEven",
    fraction=selected_signal_fraction,
    minimum_per_date=minimum_per_date,
    symbol_column="symbol",
    event_id_column="event_id",
)

selected_oof_fold_metrics_df = evaluate_signal_policy_by_fold(
    selected_oof_policy_df,
    fold_column="fold_id",
    label_column="meta_label",
    selected_column="selected_signal",
    date_column="dEven",
)

expected_fold_signal_counts = {
    int(key): int(value)
    for key, value in signal_config[
        "expected_fold_signal_counts"
    ].items()
}
actual_fold_signal_counts = (
    selected_oof_fold_metrics_df
    .set_index("fold_id")["signals"]
    .astype(int)
    .to_dict()
)

assert actual_fold_signal_counts == expected_fold_signal_counts
assert int(
    selected_oof_policy_df["selected_signal"].sum()
) == int(signal_config["expected_total_oof_signals"])

signal_policy_confirmation_df = (
    selected_oof_fold_metrics_df.loc[
        selected_oof_fold_metrics_df["fold_id"]
        .eq(signal_confirmation_fold)
    ]
    .copy()
)
signal_policy_confirmation_df["used_for_selection"] = False
signal_policy_confirmation_df["confirmation_only"] = True
signal_policy_confirmation_df["signal_fraction"] = (
    selected_signal_fraction
)

date_audit_df = (
    selected_oof_policy_df.groupby("dEven", as_index=False)
    .agg(
        fold_id=("fold_id", "first"),
        candidate_events=("event_id", "size"),
        required_quota=("daily_signal_quota", "first"),
        selected_signals=("selected_signal", "sum"),
        selected_positive_labels=(
            "meta_label",
            lambda series: int(
                series[
                    selected_oof_policy_df.loc[
                        series.index,
                        "selected_signal",
                    ].to_numpy(dtype=bool)
                ].sum()
            ),
        ),
        raw_score_cutoff=("daily_score_cutoff", "first"),
        minimum_raw_score=("probability_positive", "min"),
        maximum_raw_score=("probability_positive", "max"),
    )
)

assert date_audit_df["selected_signals"].eq(
    date_audit_df["required_quota"]
).all()
assert date_audit_df["selected_signals"].ge(1).all()

prediction_output_columns = [
    "fold_id",
    "event_id",
    "symbol",
    "dEven",
    "event_end_date",
    "meta_label",
    "probability_positive",
    "daily_candidate_count",
    "daily_rank",
    "daily_signal_quota",
    "daily_score_cutoff",
    "selected_signal",
]
selected_oof_policy_df[
    prediction_output_columns
].to_csv(
    RESULT_PATHS["predictions"]
    / "08_oof_signal_policy_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)

signal_policy_confirmation_df.to_csv(
    RESULT_PATHS["audits"]
    / "08_signal_policy_confirmation.csv",
    index=False,
    encoding="utf-8-sig",
)
date_audit_df.to_csv(
    RESULT_PATHS["audits"]
    / "08_signal_policy_date_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

display(selected_oof_fold_metrics_df)
display(signal_policy_confirmation_df)
print("Selected daily signal fraction:", selected_signal_fraction)
print(
    "Total selected OOF signals:",
    int(selected_oof_policy_df["selected_signal"].sum()),
)


## 8. Freeze the probability interpretation and signal rule

In [ ]:
policy_artifact = {
    "stage": 8,
    "status": "frozen_train_only_policy",
    "stage_version": "v1_raw_rank_daily_top_fraction",
    "score_policy": {
        "source_model": "xgboost",
        "source_prediction_column": "probability_positive",
        "selected_calibration_method": (
            selected_calibration_method
        ),
        "probability_calibrator_fitted": calibration_fitted,
        "score_name_for_downstream_use": "xgboost_ranking_score",
        "literal_probability_interpretation_allowed": False,
        "ranking_use_allowed": True,
        "reason": (
            "Post-hoc calibration candidates did not satisfy the "
            "frozen temporal stability and rank-preservation gates."
        ),
    },
    "signal_policy": {
        "policy_type": "daily_cross_sectional_top_fraction",
        "selected_fraction": selected_signal_fraction,
        "minimum_signals_per_date": minimum_per_date,
        "ranking_order": signal_config["ranking_order"],
        "fixed_probability_threshold_selected": False,
        "daily_cutoff_is_dynamic": True,
        "development_folds": sorted(development_signal_folds),
        "confirmation_fold": signal_confirmation_fold,
        "confirmation_fold_used_for_selection": False,
        "expected_total_oof_signals": int(
            selected_oof_policy_df["selected_signal"].sum()
        ),
        "fold_signal_counts": actual_fold_signal_counts,
    },
    "selection_evidence": {
        "calibration_development_folds": sorted(
            development_calibration_folds
        ),
        "calibration_confirmation_fold": confirmation_fold,
        "signal_development_folds": sorted(
            development_signal_folds
        ),
        "signal_confirmation_fold": signal_confirmation_fold,
        "economic_returns_used": False,
        "portfolio_metrics_used": False,
        "unseen_test_used": False,
    },
    "downstream_rule": (
        "For every signal date, rank all eligible long candidate events "
        "by the raw XGBoost score descending, then symbol and event ID "
        "ascending; select max(1, ceil(0.05 * candidate_count))."
    ),
}

policy_artifact_path = (
    RESULT_PATHS["manifests"]
    / "08_probability_signal_policy.json"
)
policy_artifact_path.write_text(
    json.dumps(
        policy_artifact,
        ensure_ascii=False,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

manifest = {
    "stage": 8,
    "status": "frozen_after_integrity_checks",
    "stage_version": "v1_train_only_probability_signal_policy",
    "notebook": "08_unseen_test_evaluation.ipynb",
    "git_commit_sha": git_commit_sha(REPOSITORY_ROOT),
    "software": software_manifest(),
    "lineage": {
        "stage06_code_commit_sha": stage06_manifest["git_commit_sha"],
        "stage06_study_signature": stage06_manifest["study_signature"],
        "stage06_selected_model": (
            stage06_decision["primary_selected_model"]
        ),
        "stage07_code_commit_sha": stage07_manifest["git_commit_sha"],
        "stage07_model_sha256": (
            stage07_manifest["model"]["model_sha256"]
        ),
        "stage07_training_population_sha256": (
            stage07_manifest["training_population"][
                "training_population_sha256"
            ]
        ),
    },
    "oof_input": {
        "rows": int(len(oof)),
        "fold_counts": actual_fold_counts,
        "first_signal_date": str(oof["dEven"].min()),
        "last_signal_date": str(oof["dEven"].max()),
        "selected_model": selected_model,
    },
    "calibration": {
        "selected_method": selected_calibration_method,
        "calibrator_fitted": calibration_fitted,
        "development_folds": sorted(
            development_calibration_folds
        ),
        "confirmation_fold": confirmation_fold,
        "fixed_probability_threshold_selected": False,
        "literal_probability_interpretation_allowed": False,
    },
    "signal_policy": policy_artifact["signal_policy"],
    "safeguards": {
        "unseen_test_loaded": False,
        "unseen_test_labels_used": False,
        "economic_returns_used_for_selection": False,
        "portfolio_metrics_used_for_selection": False,
        "model_retrained": False,
    },
    "configuration_hash": stable_object_hash(
        {
            "policy_config": policy_config,
            "stage06_decision": stage06_decision,
            "stage06_manifest_signature": (
                stage06_manifest["study_signature"]
            ),
            "stage07_model_sha256": (
                stage07_manifest["model"]["model_sha256"]
            ),
            "calibration_summary": (
                calibration_method_summary_df.to_dict(
                    orient="records"
                )
            ),
            "signal_policy_ranking": (
                signal_policy_ranking_df.to_dict(
                    orient="records"
                )
            ),
            "policy_artifact": policy_artifact,
        }
    ),
}

manifest_path = (
    RESULT_PATHS["manifests"]
    / "08_probability_signal_policy_manifest.json"
)
manifest_path.write_text(
    json.dumps(
        manifest,
        ensure_ascii=False,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print("Policy artifact:", policy_artifact_path)
print("Manifest:", manifest_path)
print("Manifest code SHA:", manifest["git_commit_sha"])


## 9. Final integrity checks

In [ ]:
assert selected_calibration_method == "raw_identity"
assert calibration_fitted is False
assert np.isclose(selected_signal_fraction, 0.05)
assert int(selected_oof_policy_df["selected_signal"].sum()) == 3016
assert actual_fold_signal_counts == {
    1: 561,
    2: 566,
    3: 594,
    4: 624,
    5: 671,
}
assert date_audit_df["selected_signals"].eq(
    date_audit_df["required_quota"]
).all()
assert signal_policy_confirmation_df["fold_id"].eq(5).all()
assert signal_policy_confirmation_df[
    "used_for_selection"
].eq(False).all()

assert policy_artifact["score_policy"][
    "literal_probability_interpretation_allowed"
] is False
assert policy_artifact["signal_policy"][
    "fixed_probability_threshold_selected"
] is False
assert policy_artifact["signal_policy"][
    "confirmation_fold_used_for_selection"
] is False

assert manifest["safeguards"]["unseen_test_loaded"] is False
assert manifest["safeguards"]["unseen_test_labels_used"] is False
assert manifest["safeguards"][
    "economic_returns_used_for_selection"
] is False
assert manifest["safeguards"][
    "portfolio_metrics_used_for_selection"
] is False
assert manifest["safeguards"]["model_retrained"] is False

print("Notebook 08 integrity checks: PASSED")
print("Selected calibration method: raw identity")
print("Probability calibrator fitted: False")
print("Literal probability interpretation allowed: False")
print("Score use: cross-sectional ranking only")
print("Selected signal policy: daily top 5%")
print("Minimum signals per date: 1")
print("Total selected OOF signals: 3016")
print("Development folds for signal policy: 1, 2, 3, 4")
print("Internal confirmation fold: 5")
print("Confirmation fold used for selection: False")
print("Fixed probability threshold selected: False")
print("Economic returns used for selection: False")
print("Portfolio metrics used for selection: False")
print("Unseen-test loaded: False")
print("Unseen-test labels used: False")
print(
    "Next stage: apply the frozen daily rank policy once to "
    "the untouched unseen-test candidate period."
)


## Review before freezing Stage 08

Inspect:

- `results/audits/08_oof_input_audit.csv`
- `results/audits/08_calibration_temporal_fold_metrics.csv`
- `results/audits/08_calibration_method_summary.csv`
- `results/audits/08_calibration_confirmation.csv`
- `results/audits/08_signal_policy_fold_metrics.csv`
- `results/audits/08_signal_policy_summary.csv`
- `results/audits/08_signal_policy_confirmation.csv`
- `results/audits/08_signal_policy_date_audit.csv`
- `results/manifests/08_probability_signal_policy.json`
- `results/manifests/08_probability_signal_policy_manifest.json`
- local detailed output:
  `results/predictions/08_oof_signal_policy_predictions.csv`

Do not load unseen-test data and do not proceed to the confirmatory test until
these policy artifacts are audited and frozen.
